### Importações

In [30]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, precision_recall_curve, f1_score
import xgboost as xgb
from sklearn.preprocessing import StandardScaler

### Carrega dataset

In [17]:
df = pd.read_parquet("../data/processed/flights_processed.parquet")

In [3]:
pd.set_option('display.max_columns', None) 
df.head()

,AIRLINE,YEAR,MONTH,DAY,DAY_OF_WEEK,AIRLINE_CODE,FLIGHT_NUMBER,TAIL_NUMBER,ORIGIN_AIRPORT,DESTINATION_AIRPORT,SCHEDULED_DEPARTURE,DEPARTURE_DELAY,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,ORIGIN_AIRPORT_NAME,ORIGIN_CITY,ORIGIN_STATE,ORIGIN_LAT,ORIGIN_LON,DEST_AIRPORT_NAME,DEST_CITY,DEST_STATE,DEST_LAT,DEST_LON,AIRLINE_NAME,SCHEDULED_DEPARTURE_MIN,DATE,HOUR,IS_DELAYED,IS_SHORT_DISTANCE,IS_MEDIUM_DISTANCE,IS_LONG_DISTANCE,IS_MORNING,IS_AFTERNOON,IS_NIGHT,IS_HOLIDAY,ORIGIN_AIRPORT_TOP10,DESTINATION_AIRPORT_TOP10,ORIGIN_CITY_TOP10,DEST_CITY_TOP10,ORIGIN_STATE_TOP10,DEST_STATE_TOP10,TAIL_NUMBER_TOP10,AIRLINE_TOP10,FLIGHT_NUMBER_TOP10,ROUTE
0,AS,2015,1,1,4,AS,98,N407AS,ANC,SEA,5,-11.0,205.0,1448,430,Ted Stevens Anchorage International Airport,Anchorage,AK,61.17432,-149.99619,Seattle-Tacoma International Airport,Seattle,WA,47.44898,-122.30931,Alaska Airlines Inc.,5,2015-01-01,0,0,0,1,0,1,0,0,1,Other,Other,Other,Other,Other,Other,Other,Other,Other,ANC-SEA
1,AA,2015,1,1,4,AA,2336,N3KUAA,LAX,PBI,10,-8.0,280.0,2330,750,Los Angeles International Airport,Los Angeles,CA,33.94254,-118.40807,Palm Beach International Airport,West Palm Beach,FL,26.68316,-80.09559,American Airlines Inc.,10,2015-01-01,0,0,0,0,1,1,0,0,1,LAX,Other,Los Angeles,Other,CA,FL,Other,AA,Other,LAX-PBI
2,US,2015,1,1,4,US,840,N171US,SFO,CLT,20,-2.0,286.0,2296,806,San Francisco International Airport,San Francisco,CA,37.61900,-122.37484,Charlotte Douglas International Airport,Charlotte,NC,35.21401,-80.94313,US Airways Inc.,20,2015-01-01,0,0,0,0,1,1,0,0,1,SFO,Other,San Francisco,Other,CA,Other,Other,US,Other,SFO-CLT
3,AA,2015,1,1,4,AA,258,N3HYAA,LAX,MIA,20,-5.0,285.0,2342,805,Los Angeles International Airport,Los Angeles,CA,33.94254,-118.40807,Miami International Airport,Miami,FL,25.79325,-80.29056,American Airlines Inc.,20,2015-01-01,0,0,0,0,1,1,0,0,1,LAX,Other,Los Angeles,Other,CA,FL,Other,AA,Other,LAX-MIA
4,AS,2015,1,1,4,AS,135,N527AS,SEA,ANC,25,-1.0,235.0,1448,320,Seattle-Tacoma International Airport,Seattle,WA,47.44898,-122.30931,Ted Stevens Anchorage International Airport,Anchorage,AK,61.17432,-149.99619,Alaska Airlines Inc.,25,2015-01-01,0,0,0,1,0,1,0,0,1,Other,Other,Other,Other,Other,Other,Other,Other,Other,SEA-ANC


### Seleciona features sem data leakege e nos formatos de interesse

In [19]:
remove = ["AIRLINE","YEAR", "DAY","AIRLINE_CODE", "FLIGHT_NUMBER", "TAIL_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT", "DEPARTURE_DELAY", "ORIGIN_AIRPORT_NAME", "ORIGIN_CITY",
          "ORIGIN_STATE","ORIGIN_LAT", "ORIGIN_LON","DEST_AIRPORT_NAME", "DEST_CITY", "DEST_STATE", "DEST_LAT", "DEST_LON", "SCHEDULED_DEPARTURE", "DATE", "AIRLINE_NAME"]
df = df.drop(columns=remove)

### One-Hot-Encoding

In [21]:
categorical_features = [ "MONTH", "DAY_OF_WEEK","ORIGIN_AIRPORT_TOP10","DESTINATION_AIRPORT_TOP10", "ORIGIN_CITY_TOP10", 
                        "DEST_CITY_TOP10", "ORIGIN_STATE_TOP10", "DEST_STATE_TOP10", "TAIL_NUMBER_TOP10","AIRLINE_TOP10", "FLIGHT_NUMBER_TOP10", "ROUTE_TOP10"]
df = pd.get_dummies(df, columns=categorical_features)

### Padronização

In [26]:
continuous_features = ["SCHEDULED_TIME", "DISTANCE", "SCHEDULED_ARRIVAL"]
scaler = StandardScaler()
df[continuous_features] = scaler.fit_transform(df[continuous_features])

In [36]:
df.head()

,SCHEDULED_TIME,DISTANCE,SCHEDULED_ARRIVAL,SCHEDULED_DEPARTURE_MIN,HOUR,IS_DELAYED,IS_SHORT_DISTANCE,IS_MEDIUM_DISTANCE,IS_LONG_DISTANCE,IS_MORNING,IS_AFTERNOON,IS_NIGHT,IS_HOLIDAY,MONTH_1,MONTH_2,MONTH_3,MONTH_4,MONTH_5,MONTH_6,MONTH_7,MONTH_8,MONTH_9,MONTH_11,MONTH_12,DAY_OF_WEEK_1,DAY_OF_WEEK_2,DAY_OF_WEEK_3,DAY_OF_WEEK_4,DAY_OF_WEEK_5,DAY_OF_WEEK_6,DAY_OF_WEEK_7,ORIGIN_AIRPORT_TOP10_ATL,ORIGIN_AIRPORT_TOP10_DEN,ORIGIN_AIRPORT_TOP10_DFW,ORIGIN_AIRPORT_TOP10_EWR,ORIGIN_AIRPORT_TOP10_IAH,ORIGIN_AIRPORT_TOP10_LAS,ORIGIN_AIRPORT_TOP10_LAX,ORIGIN_AIRPORT_TOP10_ORD,ORIGIN_AIRPORT_TOP10_Other,ORIGIN_AIRPORT_TOP10_PHX,ORIGIN_AIRPORT_TOP10_SFO,DESTINATION_AIRPORT_TOP10_ATL,DESTINATION_AIRPORT_TOP10_DEN,DESTINATION_AIRPORT_TOP10_DFW,DESTINATION_AIRPORT_TOP10_EWR,DESTINATION_AIRPORT_TOP10_IAH,DESTINATION_AIRPORT_TOP10_LAS,DESTINATION_AIRPORT_TOP10_LAX,DESTINATION_AIRPORT_TOP10_ORD,DESTINATION_AIRPORT_TOP10_Other,DESTINATION_AIRPORT_TOP10_PHX,DESTINATION_AIRPORT_TOP10_SFO,ORIGIN_CITY_TOP10_Atlanta,ORIGIN_CITY_TOP10_Chicago,ORIGIN_CITY_TOP10_Dallas-Fort Worth,ORIGIN_CITY_TOP10_Denver,ORIGIN_CITY_TOP10_Houston,ORIGIN_CITY_TOP10_Las Vegas,ORIGIN_CITY_TOP10_Los Angeles,ORIGIN_CITY_TOP10_New York,ORIGIN_CITY_TOP10_Other,ORIGIN_CITY_TOP10_Phoenix,ORIGIN_CITY_TOP10_San Francisco,DEST_CITY_TOP10_Atlanta,DEST_CITY_TOP10_Chicago,DEST_CITY_TOP10_Dallas-Fort Worth,DEST_CITY_TOP10_Denver,DEST_CITY_TOP10_Houston,DEST_CITY_TOP10_Las Vegas,DEST_CITY_TOP10_Los Angeles,DEST_CITY_TOP10_New York,DEST_CITY_TOP10_Other,DEST_CITY_TOP10_Phoenix,DEST_CITY_TOP10_San Francisco,ORIGIN_STATE_TOP10_AZ,ORIGIN_STATE_TOP10_CA,ORIGIN_STATE_TOP10_CO,ORIGIN_STATE_TOP10_FL,ORIGIN_STATE_TOP10_GA,ORIGIN_STATE_TOP10_IL,ORIGIN_STATE_TOP10_NC,ORIGIN_STATE_TOP10_NV,ORIGIN_STATE_TOP10_NY,ORIGIN_STATE_TOP10_Other,ORIGIN_STATE_TOP10_TX,DEST_STATE_TOP10_AZ,DEST_STATE_TOP10_CA,DEST_STATE_TOP10_CO,DEST_STATE_TOP10_FL,DEST_STATE_TOP10_GA,DEST_STATE_TOP10_IL,DEST_STATE_TOP10_NV,DEST_STATE_TOP10_NY,DEST_STATE_TOP10_Other,DEST_STATE_TOP10_TX,DEST_STATE_TOP10_VA,TAIL_NUMBER_TOP10_N361SW,TAIL_NUMBER_TOP10_N364SW,TAIL_NUMBER_TOP10_N365SW,TAIL_NUMBER_TOP10_N394SW,TAIL_NUMBER_TOP10_N637SW,TAIL_NUMBER_TOP10_N638SW,TAIL_NUMBER_TOP10_N640SW,TAIL_NUMBER_TOP10_N641SW,TAIL_NUMBER_TOP10_N646SW,TAIL_NUMBER_TOP10_N647SW,TAIL_NUMBER_TOP10_Other,AIRLINE_TOP10_AA,AIRLINE_TOP10_B6,AIRLINE_TOP10_DL,AIRLINE_TOP10_EV,AIRLINE_TOP10_MQ,AIRLINE_TOP10_NK,AIRLINE_TOP10_OO,AIRLINE_TOP10_Other,AIRLINE_TOP10_UA,AIRLINE_TOP10_US,AIRLINE_TOP10_WN,FLIGHT_NUMBER_TOP10_188,FLIGHT_NUMBER_TOP10_223,FLIGHT_NUMBER_TOP10_251,FLIGHT_NUMBER_TOP10_327,FLIGHT_NUMBER_TOP10_356,FLIGHT_NUMBER_TOP10_403,FLIGHT_NUMBER_TOP10_464,FLIGHT_NUMBER_TOP10_711,FLIGHT_NUMBER_TOP10_719,FLIGHT_NUMBER_TOP10_761,FLIGHT_NUMBER_TOP10_Other,ROUTE_TOP10_LAS-LAX,ROUTE_TOP10_LAS-SFO,ROUTE_TOP10_LAX-JFK,ROUTE_TOP10_LAX-LAS,ROUTE_TOP10_LAX-SFO,ROUTE_TOP10_ORD-DFW,ROUTE_TOP10_ORD-LAX,ROUTE_TOP10_ORD-LGA,ROUTE_TOP10_ORD-SFO,ROUTE_TOP10_Other,ROUTE_TOP10_SFO-LAX
0,0.834507,1.021800,-2.094251,5,0,0,0,1,0,1,0,0,1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False
1,1.829347,2.470147,-1.463955,10,0,0,0,0,1,1,0,0,1,True,False,False,False,False,False,False,False,False,False,False,False,False,False,True,False,False,False,False,False,False,False,False,False,True,False,False,

### Separação treino e teste

In [37]:
#separa features e target 
X = df.drop(columns=["IS_DELAYED"])
y = df["IS_DELAYED"]

In [38]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

### XGBoost

In [39]:
# Ajustando para desbalanceamento
scale = sum(y_train==0) / sum(y_train==1)  # razão de classes
model = xgb.XGBClassifier(
    scale_pos_weight=scale,
    eval_metric="logloss",
    use_label_encoder=False,
    random_state=42
)

In [40]:
# Treinar
model.fit(X_train, y_train)

c:\Users\alice\AppData\Local\Programs\Python\Python314\Lib\site-packages\xgboost\training.py:200: UserWarning: [15:15:48] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:782: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


,"objective objective: str | xgboost.sklearn._SklObjWProto | typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]] | NoneSpecify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'binary:logistic'
,"base_score base_score: float | typing.List[float] | NoneThe initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.List[xgboost.callback.TrainingCallback] | NoneList of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: float | NoneSubsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: float | NoneSubsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: float | NoneSubsample ratio of columns when constructing each tree.,None
,"device device: str | None.. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: int | None.. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: str | typing.List[str | typing.Callable] | typing.Callable | None.. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes from sklearn.metrics import mean_absolute_error X, y = load_diabetes(return_X_y=True) reg = xgb.XGBRegressor( tree_meth

In [41]:
# Previsão
y_prob = model.predict_proba(X_test)[:,1]

In [42]:
#avalia melhor threshould
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
f1_scores = 2 * (precision * recall) / (precision + recall)
best_idx = np.argmax(f1_scores)
best_threshold = thresholds[best_idx]
print("Melhor threshold:", best_threshold)

Melhor threshold: 0.53040516


In [43]:
# Ajustando threshold manual ou usando PR curve depois
threshold = 0.533
y_pred = (y_prob >= threshold).astype(int)

In [44]:
# Avaliar
print(classification_report(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_prob))

              precision    recall  f1-score   support

           0       0.88      0.70      0.78   1269932
           1       0.31      0.60      0.41    296668

    accuracy                           0.68   1566600
   macro avg       0.60      0.65      0.59   1566600
weighted avg       0.77      0.68      0.71   1566600

ROC-AUC: 0.7033599984629113


In [ ]:
#from imblearn.under_sampling import RandomUnderSampler
#from collections import Counter
#rus = RandomUnderSampler(random_state=42)
#X_res, y_res = rus.fit_resample(X, y)

#print(Counter(y))
#print(Counter(y_res))